In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("interview").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/25 17:50:57 WARN Utils: Your hostname, codespaces-9ec455, resolves to a loopback address: 127.0.0.1; using 10.0.2.101 instead (on interface eth0)
26/06/25 17:50:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 17:50:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

data = """
AAPL,150,10,2026-05-07T10:00:00
GOOG,2800,2,2026-05-07T10:05:00
AAPL,152,5,2026-05-07T10:10:00
MSFT,300,8,2026-05-07T10:15:00
AAPL,149,20,2026-05-07T10:20:00
"""

schema = StructType([
    StructField("Company", StringType()),
    StructField("Price", StringType()),
    StructField("Quantity", StringType()),
    StructField("Buy_time", StringType())
])

lines = [line.strip() for line in data.strip().split('\n') if line.strip()]

rows = [tuple(line.split(',')) for line in lines]

df = spark.createDataFrame(rows, schema)
df.show()
df.createOrReplaceTempView('stocks')
spark.sql("""
    WITH weighted_cte as (
        SELECT company, (sum(price) * sum(quantity))/sum(quantity) as weighted_avg 
        from stocks
        group by company
    ),
    ranked_cte AS(
        SELECT company, weighted_avg, dense_rank() over (order by weighted_avg desc) as r
        from weighted_cte
    )
    select company, weighted_avg, r
    from ranked_cte
    where r <= 2
""").show()

# stocks_df = spark.read.format("text").schema(schema).load("/workspaces/pyspark_udemy_codespace/data/stocks.txt")
# stocks_df.show()


# split_data = data.split()
# print(split_data)


+-------+-----+--------+-------------------+
|Company|Price|Quantity|           Buy_time|
+-------+-----+--------+-------------------+
|   AAPL|  150|      10|2026-05-07T10:00:00|
|   GOOG| 2800|       2|2026-05-07T10:05:00|
|   AAPL|  152|       5|2026-05-07T10:10:00|
|   MSFT|  300|       8|2026-05-07T10:15:00|
|   AAPL|  149|      20|2026-05-07T10:20:00|
+-------+-----+--------+-------------------+



26/06/25 17:51:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/25 17:51:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/25 17:51:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/25 17:51:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/25 17:51:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/25 17:51:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/25 1

+-------+------------+---+
|company|weighted_avg|  r|
+-------+------------+---+
|   GOOG|      2800.0|  1|
|   AAPL|       451.0|  2|
+-------+------------+---+



In [3]:
"""
A bank wants to know, for each customer, their total spend and transaction count, ordered by total spend descending. Return the top 3.
"""
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, DateType, FloatType
from datetime import date

data = """
101,2026-05-01,150.00,Amazon
102,2026-05-02,200.00,Whole_Foods
101,2026-05-03,80.00,Starbucks
103,2026-05-03,500.00,Apple
102,2026-05-04,75.00,Target
101,2026-05-05,40.00,Starbucks
104,2026-05-06,1200.00,Delta
103,2026-05-07,250.00,Best_Buy
102,2026-05-07,90.00,Whole_Foods
105,2026-05-08,30.00,CVS
"""

schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("txn_date", DateType()),
    StructField("amount", FloatType()),
    StructField("merchant", StringType())
])

lines = [line.strip() for line in data.split('\n') if line.strip()]
rows = [(
    int(parts[0]),
    date.fromisoformat(parts[1]),
    float(parts[2]),
    parts[3]
) for parts in (line.split(',') for line in lines)]

customer_df = spark.createDataFrame(rows, schema)
customer_df.createOrReplaceTempView("transactions")
spark.sql("""
    SELECT customer_id, SUM(amount) AS total_spend, COUNT(*) AS transaction_count
    FROM transactions
    GROUP BY customer_id
    ORDER BY 2 DESC
""").show()

+-----------+-----------+-----------------+
|customer_id|total_spend|transaction_count|
+-----------+-----------+-----------------+
|        104|     1200.0|                1|
|        103|      750.0|                2|
|        102|      365.0|                3|
|        101|      270.0|                3|
|        105|       30.0|                1|
+-----------+-----------+-----------------+



In [4]:
data = """customer_id,category,amount
101,FOOD,50.00
102,FOOD,80.00
101,FOOD,40.00
103,FOOD,120.00
102,FOOD,60.00
103,TRAVEL,500.00
102,TRAVEL,300.00
104,TRAVEL,800.00
101,RETAIL,200.00
102,RETAIL,150.00
103,RETAIL,80.00
104,RETAIL,500.00
105,FOOD,90.00
"""

schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("category", StringType()),
    StructField("amount", FloatType())
])

lines = [line.strip() for line in data.split('\n') if line.strip()]
header = lines[0].split(',')
rows = [(
    int(parts[0]),
    parts[1],
    float(parts[2])
) for parts in (line.split(',') for line in lines[1:])]
transactions = spark.createDataFrame(rows, schema)
transactions.show()
transactions.createOrReplaceTempView("transactions")
spark.sql("""
    WITH totals AS (
        SELECT customer_id, category, SUM(amount) AS total_category_spend, DENSE_RANK() OVER (PARTITION BY category ORDER BY SUM(amount) DESC) AS rank
        FROM transactions
        GROUP BY customer_id, category
    )
    SELECT * FROM totals WHERE rank = 1
""").show()

+-----------+--------+------+
|customer_id|category|amount|
+-----------+--------+------+
|        101|    FOOD|  50.0|
|        102|    FOOD|  80.0|
|        101|    FOOD|  40.0|
|        103|    FOOD| 120.0|
|        102|    FOOD|  60.0|
|        103|  TRAVEL| 500.0|
|        102|  TRAVEL| 300.0|
|        104|  TRAVEL| 800.0|
|        101|  RETAIL| 200.0|
|        102|  RETAIL| 150.0|
|        103|  RETAIL|  80.0|
|        104|  RETAIL| 500.0|
|        105|    FOOD|  90.0|
+-----------+--------+------+

+-----------+--------+--------------------+----+
|customer_id|category|total_category_spend|rank|
+-----------+--------+--------------------+----+
|        102|    FOOD|               140.0|   1|
|        104|  RETAIL|               500.0|   1|
|        104|  TRAVEL|               800.0|   1|
+-----------+--------+--------------------+----+



In [5]:
from pyspark.sql.types import BooleanType, DoubleType
from datetime import date

data = """
1,Alice Chen,Engineering,150000.00,2020-03-15,true
2,Bob Martinez,Sales,120000.00,2018-07-22,true
3,Carol Singh,Engineering,135000.00,2019-11-30,false
4,Dave Kim,Marketing,95000.00,2021-06-10,true
5,Eve Patel,Sales,110000.00,2022-02-14,true
6,Frank Liu,Engineering,140000.00,2017-09-05,false
"""

schema = StructType([
    StructField("emp_id", IntegerType()),
    StructField("name", StringType()),
    StructField("department", StringType()),
    StructField("salary", DoubleType()),
    StructField("hire_date", DateType()),
    StructField("is_active", BooleanType())
])

lines = [line for line in data.split('\n') if line.strip()]
rows = [(
    int(parts[0]),
    parts[1],
    parts[2],
    float(parts[3]),
    date.fromisoformat(parts[4]),
    parts[5] == 'true'
) for parts in (line.split(',') for line in lines)]
employees = spark.createDataFrame(rows, schema)
employees.show()
employees.printSchema()

+------+------------+-----------+--------+----------+---------+
|emp_id|        name| department|  salary| hire_date|is_active|
+------+------------+-----------+--------+----------+---------+
|     1|  Alice Chen|Engineering|150000.0|2020-03-15|     true|
|     2|Bob Martinez|      Sales|120000.0|2018-07-22|     true|
|     3| Carol Singh|Engineering|135000.0|2019-11-30|    false|
|     4|    Dave Kim|  Marketing| 95000.0|2021-06-10|     true|
|     5|   Eve Patel|      Sales|110000.0|2022-02-14|     true|
|     6|   Frank Liu|Engineering|140000.0|2017-09-05|    false|
+------+------------+-----------+--------+----------+---------+

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [6]:
from datetime import datetime

data = """
1,SENSOR-A,2026-05-08T10:00:00,72.5,38.2
2,SENSOR-B,2026-05-08T10:00:30,68.1,42.0
3,SENSOR-A,2026-05-08T10:01:00,73.1,38.8
4,SENSOR-C,2026-05-08T10:01:15,65.4,40.5
5,SENSOR-B,2026-05-08T10:01:30,67.9,42.3
"""

schema = StructType([
    StructField("reading_id", IntegerType()),
    StructField("sensor_id", StringType()),
    StructField("read_at", TimestampType()),
    StructField("temperature", DoubleType()),
    StructField("humidity", DoubleType())
])

lines = [line for line in data.split('\n') if line.strip()]
rows = [(
    int(parts[0]),
    parts[1],
    datetime.fromisoformat(parts[2]),
    float(parts[3]),
    float(parts[4])
) for parts in (line.split(',') for line in lines)]
sensor_data = spark.createDataFrame(rows, schema)
sensor_data.show()
sensor_data.printSchema()

+----------+---------+-------------------+-----------+--------+
|reading_id|sensor_id|            read_at|temperature|humidity|
+----------+---------+-------------------+-----------+--------+
|         1| SENSOR-A|2026-05-08 10:00:00|       72.5|    38.2|
|         2| SENSOR-B|2026-05-08 10:00:30|       68.1|    42.0|
|         3| SENSOR-A|2026-05-08 10:01:00|       73.1|    38.8|
|         4| SENSOR-C|2026-05-08 10:01:15|       65.4|    40.5|
|         5| SENSOR-B|2026-05-08 10:01:30|       67.9|    42.3|
+----------+---------+-------------------+-----------+--------+

root
 |-- reading_id: integer (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- read_at: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)



In [7]:
from pyspark.sql.types import DecimalType
from decimal import Decimal
from datetime import datetime

data = """
AAPL,150.25,500,2026-05-08T09:30:00
GOOG,2800.50,10,2026-05-08T09:30:15
MSFT,310.75,200,2026-05-08T09:30:30
AAPL,150.50,300,2026-05-08T09:30:45
TSLA,180.00,1000,2026-05-08T09:31:00
GOOG,2802.00,5,2026-05-08T09:31:15
"""

schema = StructType([
    StructField("ticker", StringType()),
    StructField("price", DecimalType(10, 2)),
    StructField("quantity", IntegerType()),
    StructField("traded_at", TimestampType())
])

lines = [line for line in data.split('\n') if line.strip()]
rows = [(
    parts[0],
    Decimal(parts[1]),
    int(parts[2]),
    datetime.fromisoformat(parts[3])
) for parts in (line.split(',') for line in lines)]

trades = spark.createDataFrame(rows, schema)
trades.show()
trades.printSchema()

+------+-------+--------+-------------------+
|ticker|  price|quantity|          traded_at|
+------+-------+--------+-------------------+
|  AAPL| 150.25|     500|2026-05-08 09:30:00|
|  GOOG|2800.50|      10|2026-05-08 09:30:15|
|  MSFT| 310.75|     200|2026-05-08 09:30:30|
|  AAPL| 150.50|     300|2026-05-08 09:30:45|
|  TSLA| 180.00|    1000|2026-05-08 09:31:00|
|  GOOG|2802.00|       5|2026-05-08 09:31:15|
+------+-------+--------+-------------------+

root
 |-- ticker: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- traded_at: timestamp (nullable = true)



In [8]:
schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("tier", StringType()),
    StructField("lifetime_spend", DecimalType(10, 2)),
    StructField("is_active", BooleanType())
])

customer_df = spark.read.format("json").option("multiline", True).schema(schema).load(path = "/workspaces/pyspark_udemy_codespace/data/customers.json")
customer_df.show()
customer_df.printSchema()

+-----------+------------+-------+--------+--------------+---------+
|customer_id|        name|   city|    tier|lifetime_spend|is_active|
+-----------+------------+-------+--------+--------------+---------+
|        101|  Alice Chen|Seattle|    GOLD|      12500.75|     true|
|        102|Bob Martinez| Boston|PLATINUM|      87340.20|     true|
|        103| Carol Singh|  Miami|  SILVER|        450.00|    false|
|        104|    Dave Kim|Chicago|    GOLD|       9800.50|     true|
|        105|   Eve Patel|Seattle|  BRONZE|        125.30|     true|
+-----------+------------+-------+--------+--------------+---------+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- lifetime_spend: decimal(10,2) (nullable = true)
 |-- is_active: boolean (nullable = true)



In [9]:
schema = StructType([
    StructField("event_id", IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("event_type", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("device", StringType())
])

events_df = spark.read.format("json").schema(schema).load(path = "/workspaces/pyspark_udemy_codespace/data/events.jsonl")
events_df.show()
events_df.printSchema()

+--------+-------+----------+-------------------+-------+
|event_id|user_id|event_type|          timestamp| device|
+--------+-------+----------+-------------------+-------+
|       1|    501|     login|2026-05-08 09:00:00| mobile|
|       2|    501|     click|2026-05-08 09:05:00| mobile|
|       3|    502|     login|2026-05-08 09:10:00|desktop|
|       4|    501|    logout|2026-05-08 09:30:00| mobile|
|       5|    503|     login|2026-05-08 09:45:00| tablet|
|       6|    502|     click|2026-05-08 09:50:00|desktop|
+--------+-------+----------+-------------------+-------+

root
 |-- event_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- device: string (nullable = true)



In [10]:
branches_schema = StructType([
    StructField("branch_id", IntegerType()),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType())
])

managers_schema = StructType([
    StructField("branch_id", IntegerType()),
    StructField("manager_name", StringType()),
    StructField("hire_date", DateType())
])

branches_df = spark.read.format("csv").option("header", True).schema(branches_schema).load(path = "/workspaces/pyspark_udemy_codespace/data/branches.csv")
branches_df.show()
branches_df.createOrReplaceTempView("branches")
# branches_df.printSchema()

managers_df = spark.read.format("json").schema(managers_schema).load(path = "/workspaces/pyspark_udemy_codespace/data/managers.jsonl")
managers_df.show()
managers_df.createOrReplaceTempView("managers")
# managers_df.printSchema()

spark.sql("""
    SELECT b.branch_id, b.name as branch_name, b.city, b.state, m.manager_name, m.hire_date
    FROM branches b LEFT JOIN managers m
    ON b.branch_id = m.branch_id
    ORDER BY b.branch_id
""").show()


+---------+----------------+-------+-----+
|branch_id|            name|   city|state|
+---------+----------------+-------+-----+
|        1|Downtown Seattle|Seattle|   WA|
|        2|   Boston Common| Boston|   MA|
|        3|     Miami Beach|  Miami|   FL|
|        4|     Loop Branch|Chicago|   IL|
|        5|    Capitol Hill|Seattle|   WA|
+---------+----------------+-------+-----+

+---------+------------+----------+
|branch_id|manager_name| hire_date|
+---------+------------+----------+
|        1|  Alice Chen|2018-04-15|
|        2|Bob Martinez|2020-09-01|
|        3| Carol Singh|2017-11-20|
|        4|    Dave Kim|2021-06-08|
+---------+------------+----------+

+---------+----------------+-------+-----+------------+----------+
|branch_id|     branch_name|   city|state|manager_name| hire_date|
+---------+----------------+-------+-----+------------+----------+
|        1|Downtown Seattle|Seattle|   WA|  Alice Chen|2018-04-15|
|        2|   Boston Common| Boston|   MA|Bob Martinez|

In [11]:
from pyspark.sql.types import ArrayType

address_schema = StructType([
    StructField("street", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("zip", StringType())
])

account_schema = StructType([
    StructField("account_id", StringType()),
    StructField("type", StringType()),
    StructField("balance", DecimalType(10, 2))
])

customer_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("name", StringType()),
    StructField("address", address_schema),
    StructField("accounts", ArrayType(account_schema))
])

customer_df = spark.read.format("json").schema(customer_schema).load(path = "/workspaces/pyspark_udemy_codespace/data/customer_nested_json.jsonl")
customer_df.show()
customer_df.printSchema()

+-----------+------------+--------------------+--------------------+
|customer_id|        name|             address|            accounts|
+-----------+------------+--------------------+--------------------+
|        101|  Alice Chen|{123 Main St, Sea...|[{ACC001, CHECKIN...|
|        102|Bob Martinez|{456 Oak Ave, Bos...|[{ACC003, CHECKIN...|
|        103| Carol Singh|{789 Pine Rd, Mia...|[{ACC004, CHECKIN...|
|        104|    Dave Kim|{321 Elm Way, Chi...|                  []|
+-----------+------------+--------------------+--------------------+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |-- accounts: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- account_id: string (nullable = true)
 |    |    |-- typ

In [12]:
schema = StructType([
    StructField("txn_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("txn_date", DateType()),
    StructField("amount", FloatType()),
    StructField("merchant", StringType())
])

# "/workspaces/pyspark_udemy_codespace/data/transactions.csv"
transactions_df = spark.read.format("csv")\
                            .option("header", True)\
                            .schema(schema)\
                            .load(path = "/workspaces/pyspark_udemy_codespace/data/transactions.csv")

transactions_df.show()
transactions_df.createOrReplaceTempView("transactions")

spark.sql("""
    WITH cte AS(
        SELECT customer_id, TRUNC(txn_date, 'MM') AS month, SUM(amount) AS monthly_spend, COUNT(*) AS txn_count, AVG(amount) AS avg_txn_amount
          -- DENSE_RANK() OVER (PARTITION BY customer_id, TRUNC(txn_date, 'MM') ORDER BY SUM(amount) DESC) AS monthly_rank,
          -- SUM(amount) OVER (PARTITION BY customer_id, TRUNC(txn_date, 'MM') ORDER BY txn_date) AS cumulative_spend
        FROM transactions
        GROUP BY customer_id, TRUNC(txn_date, 'MM')
    )
    SELECT c.*, DENSE_RANK() OVER (PARTITION BY month ORDER BY monthly_spend DESC, customer_id ASC) AS monthly_rank, 
        SUM(monthly_spend) OVER (PARTITION BY customer_id ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_spend
    FROM cte c
    ORDER BY customer_id, month
""").show()

+------+-----------+----------+------+-----------+
|txn_id|customer_id|  txn_date|amount|   merchant|
+------+-----------+----------+------+-----------+
|     1|        101|2024-01-05| 250.0|Whole_Foods|
|     2|        102|2024-01-08|  75.0|  Starbucks|
|     3|        101|2024-01-15| 180.0|     Amazon|
|     4|        103|2024-01-20| 800.0|      Apple|
|     5|        102|2024-01-22|  55.0|     Target|
|     6|        101|2024-01-28| 220.0|   Best_Buy|
|     7|        104|2024-02-02|  30.0|        CVS|
|     8|        105|2024-02-05|1500.0|    Tiffany|
|     9|        101|2024-02-10| 300.0|      Apple|
|    10|        103|2024-02-12| 650.0|   Best_Buy|
|    11|        102|2024-02-18|  40.0|  Starbucks|
|    12|        106|2024-02-20|2200.0|      Rolex|
|    13|        104|2024-02-22|  25.0|        CVS|
|    14|        101|2024-02-28| 420.0|Whole_Foods|
|    15|        105|2024-03-03| 950.0|      Apple|
|    16|        103|2024-03-08| 300.0|     Amazon|
|    17|        102|2024-03-12|

In [21]:
"""
🟡 Problem 1: Find the total revenue (quantity * price) per category, 
and the customer who spent the most overall. Return category-wise revenue sorted descending, 
plus a separate result for top spender.
"""
from pyspark.sql.functions import col, sum

schema  = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("product", StringType()),
    StructField("category", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price", FloatType()),
    StructField("order_date", DateType())
])

orders_df = spark.read.format("csv").option("header", True).schema(schema).load(path = "/workspaces/pyspark_udemy_codespace/data/e_commerce_orders.csv")

orders_metric_df = orders_df.withColumn("revenue", col("quantity") * col("price"))

total_category_revenue_df = orders_metric_df.groupBy(col("category"))\
                                            .agg(sum(col("revenue")).alias("total_revenue"))

customers_spend_df = orders_metric_df.groupBy(col("customer_name"))\
                                     .agg(sum(col("revenue")).alias("total_spent"))

total_category_revenue_df.show()
customers_spend_df.show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|     179000.0|
|  Furniture|      52000.0|
+-----------+-------------+

+-------------+-----------+
|customer_name|total_spent|
+-------------+-----------+
|        Diana|    11500.0|
|      Charlie|    83000.0|
|          Bob|    25000.0|
|        Alice|    86500.0|
|          Eve|    25000.0|
+-------------+-----------+



In [14]:
"""
🟡 Problem 2: For each user_id, 
calculate the time difference (in minutes) between their login and logout events. 
Only include users who have both events.
"""
from pyspark.sql.functions import unix_timestamp

user_activity_schema = StructType([
    StructField("user_id", IntegerType()),
    StructField("action", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("device", StringType()),
    StructField("amount", FloatType())
])

user_activity_df = spark.read.format("json")\
                             .schema(user_activity_schema)\
                             .load(path = "/workspaces/pyspark_udemy_codespace/data/user_activity_logs.jsonl")

login_df = user_activity_df.filter(col("action") == "login").alias("in")
logout_df = user_activity_df.filter(col("action") == "logout").alias("out")

login_logout_join = col("in.user_id") == col("out.user_id")
time_diff_df = login_df.join(logout_df, login_logout_join, "inner")\
                       .withColumn("time_difference", (unix_timestamp(col("out.timestamp")) - unix_timestamp(col("in.timestamp"))) / 60)\
                       .select([
                           col("in.user_id").alias("user_id"),
                           col("time_difference")
                       ])
time_diff_df.show()

+-------+---------------+
|user_id|time_difference|
+-------+---------------+
|    101|           75.0|
|    102|          115.0|
+-------+---------------+



In [26]:
"""
🟡 Problem 3: Flatten this DataFrame so each row represents one (employee, skill) pair — include name, department, city, skill, and years. 
Then find the most common skill per department.
"""
from pyspark.sql.functions import explode, count

schema = StructType([
    StructField("emp_id", IntegerType()),
    StructField("name", StringType()),
    StructField("department", StringType()),
    StructField("address", StructType([
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("pin", StringType())
    ])),
    StructField("skills", ArrayType(StructType([
        StructField("skill", StringType()),
        StructField("years", IntegerType())
    ])))
])

company_employee_records_df = spark.read.format("json")\
                                        .option("multiline", True)\
                                        .schema(schema)\
                                        .load(path = "/workspaces/pyspark_udemy_codespace/data/company_employee_records.json")

exploded_df = company_employee_records_df.select(
    col("name"),
    col("department"),
    col("address.city").alias("city"),
    explode(col("skills")).alias("struct_skills")
).select(
    "name",
    "department",
    "city",
    col("struct_skills.skill").alias("skill"),
    col("struct_skills.years").alias("years")
)

result_df = exploded_df.groupBy(col("department"), col("skill")).agg(count("*").alias("skill_count"))
result_df.show()

+-----------+-----------+-----------+
| department|      skill|skill_count|
+-----------+-----------+-----------+
|Engineering|     Python|          1|
|      Sales|Negotiation|          1|
|Engineering|      Spark|          1|
|  Marketing|        SEO|          1|
|Engineering|       Java|          1|
|Engineering|      Kafka|          1|
+-----------+-----------+-----------+

